
# Low-σ 先验样本诊断：FE 重构精度与 SNO 预测精度

本 notebook 用于诊断规则圆环 fixed-annulus SNO 在低频 PI-sampler 样本上的表现。

目标：

1. 分别从 \(\sigma=0.5,1.0,2.0\) 构造样本；
2. 检查 FE oracle reconstruction 精度；
3. 检查 SNO zero-shot prediction 精度；
4. 导出 `.mat` 文件，方便后续 MATLAB 可视化。

诊断逻辑：

- 若 FE 重构误差小而 SNO 预测误差大，说明 FE 表示空间基本够，问题更可能在 Transformer 的低频 OOD 映射。
- 若 FE 重构误差也大，说明当前 FE basis / branch / normalizer 对低频样本覆盖不足。


In [ ]:

# ============================================================
# 0. 可选：JAX/GPU 环境设置
#    注意：这些环境变量最好在第一次 import jax 之前设置。
# ============================================================
import os

# 按你的机器选择 GPU；不需要指定时可以注释掉。
# os.environ["CUDA_VISIBLE_DEVICES"] = "5"

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
# os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.75")



## 1. 用户配置区

下面这一格是最常修改的部分。请确保：

- `PROJECT_DIR` 指向包含 `config.py / data.py / models.py / train.py` 的 fixed-annulus SNO 代码目录；
- `OUT_DIR` 和 `RUN_NAME` 指向已训练参数所在目录；
- 架构参数必须与训练时一致，例如 `n_basis`、`theta_size`、`radial_size`、`trunk_width`、`transformer_dim` 等。


In [ ]:

from pathlib import Path

# ============================================================
# 1. 路径配置
# ============================================================
PROJECT_DIR = "/home/user/data/Hollon/海洋工程水动力/annulus_sno_annulus_only_v2/annulus_sno_annulus_only_v2"
OUT_DIR = "/home/user/data/Hollon/海洋工程水动力/annulus_sno_annulus_only_v2/out"
RUN_NAME = "test"

# 参数文件名。设为 "auto" 时会按常见文件名自动搜索。
FE_PARAM_NAME = "auto"   # e.g. "fe_params.msgpack" or "fe_params_physv2.msgpack"
OL_PARAM_NAME = "auto"   # e.g. "ol_params.msgpack" or "ol_params_physv2.msgpack"

# 输出 mat 文件；设为 None 时保存到 <OUT_DIR>/<RUN_NAME>/low_sigma_fe_sno_diagnostic.mat
MAT_PATH = None

# ============================================================
# 2. 低频 sigma 诊断配置
# ============================================================
SIGMAS = [0.5, 1.0, 2.0]
N_PER_SIGMA = 128
SEED = 0
EXPORT_GRID_FIELDS = True

# ============================================================
# 3. 几何与 PDE 参数：必须与你训练/推理约定一致
# ============================================================
R_INNER = 0.2
R_OUTER = 1.0
K_MIN = 1.0
K_MAX = 1.0

# ============================================================
# 4. 采样与 FE/OL 架构参数：必须与训练时一致
# ============================================================
N_BASIS = 512
THETA_SIZE = 128
RADIAL_SIZE = 32
RANDOM_PROBE_POINTS = 1024
POD_SNAPSHOTS = 100

TRUNK_WIDTH = 512
TRUNK_DEPTH = 5
CNN_DENSE_WIDTH = 1024

TRANSFORMER_DIM = 512
TRANSFORMER_HEADS = 8
TRANSFORMER_LAYERS = 4
TRANSFORMER_MLP_DIM = 1024
SEQ_CHUNKS = 32
COND_CHUNKS = 32

# TrainState 初始化需要这些字段；这里不进行训练。
FE_STEPS = 300_000
OL_STEPS = 200_000
FE_LR = 1e-3
OL_LR = 1e-3
WEIGHT_DECAY = 1e-6



## 2. 导入 fixed-annulus SNO 模块

优先导入项目中的 `config.py / data.py / models.py / train.py`。如果你的文件名是上传版本中的 `config_annulus.py` 等，代码也提供了 fallback。


In [ ]:

import sys
import copy
from typing import Iterable

import numpy as np
from scipy.io import savemat

project_path = Path(PROJECT_DIR).expanduser().resolve()
if not project_path.exists():
    raise FileNotFoundError(f"PROJECT_DIR does not exist: {project_path}")
sys.path.insert(0, str(project_path))


def import_annulus_modules():
    """Import fixed-annulus SNO project modules."""
    try:
        import jax
        import jax.numpy as jnp
        from flax import serialization

        from config import AnnulusConfig
        from data import (
            make_condition_tokens,
            make_source_tokens,
            normalize_f,
            normalize_u,
            denormalize_f,
            denormalize_u,
            sample_batch,
        )
        from models import FunctionEncoder
        from train import (
            create_fe_state,
            create_ol_state,
            load_field_normalizer,
            rl2_error,
        )
        return {
            "AnnulusConfig": AnnulusConfig,
            "make_condition_tokens": make_condition_tokens,
            "make_source_tokens": make_source_tokens,
            "normalize_f": normalize_f,
            "normalize_u": normalize_u,
            "denormalize_f": denormalize_f,
            "denormalize_u": denormalize_u,
            "sample_batch": sample_batch,
            "FunctionEncoder": FunctionEncoder,
            "create_fe_state": create_fe_state,
            "create_ol_state": create_ol_state,
            "load_field_normalizer": load_field_normalizer,
            "rl2_error": rl2_error,
            "serialization": serialization,
            "jax": jax,
            "jnp": jnp,
        }
    except ImportError:
        # Fallback for files named config_annulus.py / data_annulus.py / models_annulus.py / train_annulus.py.
        import jax
        import jax.numpy as jnp
        from flax import serialization

        import config_annulus as config_mod
        sys.modules.setdefault("config", config_mod)

        import data_annulus as data_mod
        sys.modules.setdefault("data", data_mod)

        import models_annulus as models_mod
        sys.modules.setdefault("models", models_mod)

        import train_annulus as train_mod

        return {
            "AnnulusConfig": config_mod.AnnulusConfig,
            "make_condition_tokens": data_mod.make_condition_tokens,
            "make_source_tokens": data_mod.make_source_tokens,
            "normalize_f": data_mod.normalize_f,
            "normalize_u": data_mod.normalize_u,
            "denormalize_f": data_mod.denormalize_f,
            "denormalize_u": data_mod.denormalize_u,
            "sample_batch": data_mod.sample_batch,
            "FunctionEncoder": models_mod.FunctionEncoder,
            "create_fe_state": train_mod.create_fe_state,
            "create_ol_state": train_mod.create_ol_state,
            "load_field_normalizer": train_mod.load_field_normalizer,
            "rl2_error": train_mod.rl2_error,
            "serialization": serialization,
            "jax": jax,
            "jnp": jnp,
        }


MODS = import_annulus_modules()
jax = MODS["jax"]
jnp = MODS["jnp"]

print("JAX devices:", jax.devices())



## 3. 构造配置并加载训练好的 FE / OL 参数


In [ ]:

def to_numpy(x):
    return np.asarray(jax.device_get(x))


def find_param_file(output_dir: Path, explicit_name: str | None, candidates: Iterable[str]) -> Path:
    """Find a saved msgpack parameter file."""
    if explicit_name and explicit_name.lower() != "auto":
        path = output_dir / explicit_name
        if not path.exists():
            raise FileNotFoundError(f"Cannot find parameter file: {path}")
        return path

    for name in candidates:
        path = output_dir / name
        if path.exists():
            return path

    tried = "\n".join(str(output_dir / name) for name in candidates)
    raise FileNotFoundError(f"Cannot find parameter file. Tried:\n{tried}")


def build_config():
    AnnulusConfig = MODS["AnnulusConfig"]
    cfg = AnnulusConfig()

    cfg.out_dir = OUT_DIR
    cfg.run_name = RUN_NAME

    # Geometry / PDE
    cfg.r_inner = R_INNER
    cfg.r_outer = R_OUTER
    cfg.k_min = K_MIN
    cfg.k_max = K_MAX

    # Sampling / discretization
    cfg.n_basis = N_BASIS
    cfg.theta_size = THETA_SIZE
    cfg.radial_size = RADIAL_SIZE
    cfg.random_probe_points = RANDOM_PROBE_POINTS
    cfg.pod_snapshots = POD_SNAPSHOTS

    # FE architecture
    cfg.trunk_width = TRUNK_WIDTH
    cfg.trunk_depth = TRUNK_DEPTH
    cfg.cnn_dense_width = CNN_DENSE_WIDTH

    # Transformer architecture
    cfg.transformer_dim = TRANSFORMER_DIM
    cfg.transformer_heads = TRANSFORMER_HEADS
    cfg.transformer_layers = TRANSFORMER_LAYERS
    cfg.transformer_mlp_dim = TRANSFORMER_MLP_DIM
    cfg.seq_chunks = SEQ_CHUNKS
    cfg.cond_chunks = COND_CHUNKS

    # Training metadata for TrainState initialization only
    cfg.fe_steps = FE_STEPS
    cfg.ol_steps = OL_STEPS
    cfg.fe_lr = FE_LR
    cfg.ol_lr = OL_LR
    cfg.weight_decay = WEIGHT_DECAY
    cfg.seed = SEED

    # Evaluation: one sigma at a time, exactly N_PER_SIGMA samples.
    cfg.num_repeats = 1
    cfg.sample_size = N_PER_SIGMA

    return cfg


cfg = build_config()
print("Output dir:", cfg.output_dir)
print("Sigmas:", SIGMAS)
print("N_PER_SIGMA:", N_PER_SIGMA)


In [ ]:

def load_trained_states(config):
    serialization = MODS["serialization"]
    create_fe_state = MODS["create_fe_state"]
    create_ol_state = MODS["create_ol_state"]
    load_field_normalizer = MODS["load_field_normalizer"]

    output_dir = config.output_dir
    normalizer = load_field_normalizer(output_dir)

    key = jax.random.PRNGKey(SEED + 20260623)
    key_fe, key_ol = jax.random.split(key, 2)

    fe_state, _ = create_fe_state(config, key_fe)
    ol_state, _ = create_ol_state(config, key_ol)

    fe_path = find_param_file(
        output_dir,
        FE_PARAM_NAME,
        candidates=(
            "fe_params_physv2.msgpack",
            "fe_params_phys.msgpack",
            "fe_params.msgpack",
        ),
    )
    ol_path = find_param_file(
        output_dir,
        OL_PARAM_NAME,
        candidates=(
            "ol_params_physv2.msgpack",
            "ol_params.msgpack",
            "transformer_params.msgpack",
        ),
    )

    fe_params = serialization.from_bytes(fe_state.params, fe_path.read_bytes())
    ol_params = serialization.from_bytes(ol_state.params, ol_path.read_bytes())

    fe_state = fe_state.replace(params=fe_params)
    ol_state = ol_state.replace(params=ol_params)

    print("[Loaded] FE params:", fe_path)
    print("[Loaded] OL params:", ol_path)
    print(
        "[Loaded normalizer] "
        f"mean_u={float(normalizer.mean_u):.6e}, std_u={float(normalizer.std_u):.6e}, "
        f"mean_f={float(normalizer.mean_f):.6e}, std_f={float(normalizer.std_f):.6e}"
    )
    return fe_state, ol_state, normalizer, fe_path, ol_path


fe_state, ol_state, normalizer, fe_path, ol_path = load_trained_states(cfg)



## 4. 定义单个 \(\sigma\) 的诊断函数

这一部分同时计算：

- FE 对 \(u\) 和 \(f\) 的 oracle reconstruction；
- SNO 预测 \(u\)；
- latent error；
- 源项、边界通量的范数，用于判断条件分布偏移。


In [ ]:

def evaluate_one_sigma(cfg_base, sigma_value: float, key, fe_state, ol_state, normalizer):
    sample_batch = MODS["sample_batch"]
    normalize_u = MODS["normalize_u"]
    normalize_f = MODS["normalize_f"]
    denormalize_u = MODS["denormalize_u"]
    denormalize_f = MODS["denormalize_f"]
    make_source_tokens = MODS["make_source_tokens"]
    make_condition_tokens = MODS["make_condition_tokens"]
    FunctionEncoder = MODS["FunctionEncoder"]
    rl2_error = MODS["rl2_error"]

    # 单独设定 sigma，保证该 batch 的 sigma 标签准确。
    cfg_eval = copy.deepcopy(cfg_base)
    cfg_eval.sigma_list = (float(sigma_value),)
    cfg_eval.num_repeats = 1
    cfg_eval.sample_size = N_PER_SIGMA

    batch = sample_batch(key, cfg_eval)

    # ============================================================
    # 1. FE encode true u and true f from regular POD grid
    # ============================================================
    u_pod_norm = normalize_u(batch.u_pod, normalizer)
    f_pod_norm = normalize_f(batch.f_pod, normalizer)

    target_u_latent = fe_state.apply_fn(
        {"params": fe_state.params},
        u_pod_norm,
        method=FunctionEncoder.encode_u,
    )
    latent_f = fe_state.apply_fn(
        {"params": fe_state.params},
        f_pod_norm,
        method=FunctionEncoder.encode_f,
    )

    # ============================================================
    # 2. FE oracle reconstruction on POD and probe points
    # ============================================================
    u_fe_pod_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        target_u_latent,
        batch.pod_coords,
        method=FunctionEncoder.reconstruct,
    )
    u_fe_probe_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        target_u_latent,
        batch.probe_coords,
        method=FunctionEncoder.reconstruct,
    )
    f_fe_pod_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        latent_f,
        batch.pod_coords,
        method=FunctionEncoder.reconstruct,
    )
    f_fe_probe_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        latent_f,
        batch.probe_coords,
        method=FunctionEncoder.reconstruct,
    )

    u_fe_pod = denormalize_u(u_fe_pod_norm, normalizer)
    u_fe_probe = denormalize_u(u_fe_probe_norm, normalizer)
    f_fe_pod = denormalize_f(f_fe_pod_norm, normalizer)
    f_fe_probe = denormalize_f(f_fe_probe_norm, normalizer)

    # ============================================================
    # 3. SNO prediction: latent_f + boundary tokens + k -> latent_u
    # ============================================================
    f_tokens = make_source_tokens(latent_f, cfg_eval)
    cond_tokens = make_condition_tokens(batch, cfg_eval)

    pred_u_latent = ol_state.apply_fn(
        {"params": ol_state.params},
        f_tokens,
        cond_tokens,
        batch.k_values,
    )

    u_sno_pod_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        pred_u_latent,
        batch.pod_coords,
        method=FunctionEncoder.reconstruct,
    )
    u_sno_probe_norm = fe_state.apply_fn(
        {"params": fe_state.params},
        pred_u_latent,
        batch.probe_coords,
        method=FunctionEncoder.reconstruct,
    )

    u_sno_pod = denormalize_u(u_sno_pod_norm, normalizer)
    u_sno_probe = denormalize_u(u_sno_probe_norm, normalizer)

    # ============================================================
    # 4. Error diagnostics
    # ============================================================
    err_fe_u_pod = rl2_error(u_fe_pod, batch.u_pod)
    err_fe_u_probe = rl2_error(u_fe_probe, batch.u_probe)
    err_fe_f_pod = rl2_error(f_fe_pod, batch.f_pod)
    err_fe_f_probe = rl2_error(f_fe_probe, batch.f_probe)

    err_sno_u_pod = rl2_error(u_sno_pod, batch.u_pod)
    err_sno_u_probe = rl2_error(u_sno_probe, batch.u_probe)
    err_latent = rl2_error(pred_u_latent, target_u_latent)

    norm_u_pod = jnp.linalg.norm(batch.u_pod, axis=-1)
    norm_f_pod = jnp.linalg.norm(batch.f_pod, axis=-1)
    norm_flux = jnp.linalg.norm(batch.boundary_flux, axis=-1)

    print(
        f"[sigma={sigma_value:.3g}] "
        f"FE_u_probe={float(err_fe_u_probe.mean()):.4e}, "
        f"FE_f_probe={float(err_fe_f_probe.mean()):.4e}, "
        f"SNO_u_probe={float(err_sno_u_probe.mean()):.4e}, "
        f"latent={float(err_latent.mean()):.4e}"
    )

    return {
        "sigma": float(sigma_value),
        "batch": batch,
        "latent_f": latent_f,
        "target_u_latent": target_u_latent,
        "pred_u_latent": pred_u_latent,
        "u_fe_pod": u_fe_pod,
        "u_fe_probe": u_fe_probe,
        "f_fe_pod": f_fe_pod,
        "f_fe_probe": f_fe_probe,
        "u_sno_pod": u_sno_pod,
        "u_sno_probe": u_sno_probe,
        "err_fe_u_pod": err_fe_u_pod,
        "err_fe_u_probe": err_fe_u_probe,
        "err_fe_f_pod": err_fe_f_pod,
        "err_fe_f_probe": err_fe_f_probe,
        "err_sno_u_pod": err_sno_u_pod,
        "err_sno_u_probe": err_sno_u_probe,
        "err_latent": err_latent,
        "norm_u_pod": norm_u_pod,
        "norm_f_pod": norm_f_pod,
        "norm_flux": norm_flux,
    }



## 5. 运行低频 \(\sigma\)-sweep 诊断


In [ ]:

key = jax.random.PRNGKey(SEED + 20260623)
sigma_keys = jax.random.split(key, len(SIGMAS))

results = []
for sigma_value, key_sigma in zip(SIGMAS, sigma_keys):
    result = evaluate_one_sigma(
        cfg_base=cfg,
        sigma_value=float(sigma_value),
        key=key_sigma,
        fe_state=fe_state,
        ol_state=ol_state,
        normalizer=normalizer,
    )
    results.append(result)

print("Done.")



## 6. 汇总误差表

重点看：

- `mean_err_fe_u_probe`：FE 对解场 \(u\) 的 oracle 重构误差；
- `mean_err_sno_u_probe`：完整 SNO 预测误差；
- `mean_err_latent`：Transformer latent prediction 误差。


In [ ]:

def make_summary(results):
    columns = [
        "sigma",
        "mean_err_fe_u_pod",
        "mean_err_fe_u_probe",
        "mean_err_fe_f_pod",
        "mean_err_fe_f_probe",
        "mean_err_sno_u_pod",
        "mean_err_sno_u_probe",
        "mean_err_latent",
        "max_err_fe_u_probe",
        "max_err_fe_f_probe",
        "max_err_sno_u_probe",
        "max_err_latent",
        "mean_norm_u_pod",
        "mean_norm_f_pod",
        "mean_norm_boundary_flux",
    ]

    summary = np.zeros((len(results), len(columns)), dtype=np.float64)
    for i, r in enumerate(results):
        summary[i, :] = np.array([
            r["sigma"],
            float(to_numpy(r["err_fe_u_pod"]).mean()),
            float(to_numpy(r["err_fe_u_probe"]).mean()),
            float(to_numpy(r["err_fe_f_pod"]).mean()),
            float(to_numpy(r["err_fe_f_probe"]).mean()),
            float(to_numpy(r["err_sno_u_pod"]).mean()),
            float(to_numpy(r["err_sno_u_probe"]).mean()),
            float(to_numpy(r["err_latent"]).mean()),
            float(to_numpy(r["err_fe_u_probe"]).max()),
            float(to_numpy(r["err_fe_f_probe"]).max()),
            float(to_numpy(r["err_sno_u_probe"]).max()),
            float(to_numpy(r["err_latent"]).max()),
            float(to_numpy(r["norm_u_pod"]).mean()),
            float(to_numpy(r["norm_f_pod"]).mean()),
            float(to_numpy(r["norm_flux"]).mean()),
        ], dtype=np.float64)
    return summary, columns


summary, summary_columns = make_summary(results)
summary_columns_np = np.array(summary_columns, dtype=object)

try:
    import pandas as pd
    summary_df = pd.DataFrame(summary, columns=summary_columns)
    display(summary_df)
except Exception:
    print(summary_columns)
    print(summary)



## 7. 导出 `.mat` 文件

导出后，你可以直接在 MATLAB 中读取并绘制：真值、FE oracle、SNO prediction、误差场。


In [ ]:

def stack_result(results, key: str):
    return np.stack([to_numpy(r[key]) for r in results], axis=0)


def stack_batch_field(results, attr: str):
    return np.stack([to_numpy(getattr(r["batch"], attr)) for r in results], axis=0)


def export_mat(config, results, fe_path: Path, ol_path: Path):
    out_dir = config.output_dir
    mat_path = Path(MAT_PATH) if MAT_PATH is not None else out_dir / "low_sigma_fe_sno_diagnostic.mat"
    mat_path.parent.mkdir(parents=True, exist_ok=True)

    Nr = config.radial_size
    Nt = config.theta_size
    n_sigma = len(results)
    B = N_PER_SIGMA

    pod_coords = to_numpy(results[0]["batch"].pod_coords)
    probe_coords = to_numpy(results[0]["batch"].probe_coords)
    x_grid = pod_coords[:, 0].reshape(Nr, Nt)
    y_grid = pod_coords[:, 1].reshape(Nr, Nt)

    sigma_values = np.asarray([r["sigma"] for r in results], dtype=np.float64)

    mat_data = {
        # Metadata
        "sigma_values": sigma_values[:, None],
        "n_sigma": np.array([[n_sigma]], dtype=np.int32),
        "n_per_sigma": np.array([[B]], dtype=np.int32),
        "summary": summary,
        "summary_columns": summary_columns_np,
        "fe_param_path": np.array(str(fe_path), dtype=object),
        "ol_param_path": np.array(str(ol_path), dtype=object),

        # Coordinates
        "pod_coords": pod_coords,
        "probe_coords": probe_coords,
        "x_grid": x_grid,
        "y_grid": y_grid,

        # True fields: [n_sigma, B, N]
        "u_true_pod": stack_batch_field(results, "u_pod"),
        "f_true_pod": stack_batch_field(results, "f_pod"),
        "u_true_probe": stack_batch_field(results, "u_probe"),
        "f_true_probe": stack_batch_field(results, "f_probe"),
        "boundary_coords": stack_batch_field(results, "boundary_coords"),
        "boundary_flux": stack_batch_field(results, "boundary_flux"),
        "k_values": stack_batch_field(results, "k_values"),

        # FE oracle reconstructions
        "u_fe_pod": stack_result(results, "u_fe_pod"),
        "u_fe_probe": stack_result(results, "u_fe_probe"),
        "f_fe_pod": stack_result(results, "f_fe_pod"),
        "f_fe_probe": stack_result(results, "f_fe_probe"),

        # SNO predictions
        "u_sno_pod": stack_result(results, "u_sno_pod"),
        "u_sno_probe": stack_result(results, "u_sno_probe"),

        # Latents
        "latent_f": stack_result(results, "latent_f"),
        "target_u_latent": stack_result(results, "target_u_latent"),
        "pred_u_latent": stack_result(results, "pred_u_latent"),

        # Per-sample errors: [n_sigma, B]
        "err_fe_u_pod": stack_result(results, "err_fe_u_pod"),
        "err_fe_u_probe": stack_result(results, "err_fe_u_probe"),
        "err_fe_f_pod": stack_result(results, "err_fe_f_pod"),
        "err_fe_f_probe": stack_result(results, "err_fe_f_probe"),
        "err_sno_u_pod": stack_result(results, "err_sno_u_pod"),
        "err_sno_u_probe": stack_result(results, "err_sno_u_probe"),
        "err_latent": stack_result(results, "err_latent"),

        # Distribution-shift helper statistics
        "norm_u_pod": stack_result(results, "norm_u_pod"),
        "norm_f_pod": stack_result(results, "norm_f_pod"),
        "norm_boundary_flux": stack_result(results, "norm_flux"),

        # Config snapshot
        "r_inner": np.array([[config.r_inner]], dtype=np.float64),
        "r_outer": np.array([[config.r_outer]], dtype=np.float64),
        "k_min": np.array([[config.k_min]], dtype=np.float64),
        "k_max": np.array([[config.k_max]], dtype=np.float64),
        "radial_size": np.array([[config.radial_size]], dtype=np.int32),
        "theta_size": np.array([[config.theta_size]], dtype=np.int32),
        "random_probe_points": np.array([[config.random_probe_points]], dtype=np.int32),
        "n_basis": np.array([[config.n_basis]], dtype=np.int32),
    }

    # Optional grid-shaped fields for MATLAB plotting.
    if EXPORT_GRID_FIELDS:
        mat_data.update({
            "u_true_grid": mat_data["u_true_pod"].reshape(n_sigma, B, Nr, Nt),
            "f_true_grid": mat_data["f_true_pod"].reshape(n_sigma, B, Nr, Nt),
            "u_fe_grid": mat_data["u_fe_pod"].reshape(n_sigma, B, Nr, Nt),
            "f_fe_grid": mat_data["f_fe_pod"].reshape(n_sigma, B, Nr, Nt),
            "u_sno_grid": mat_data["u_sno_pod"].reshape(n_sigma, B, Nr, Nt),
            "u_fe_abs_error_grid": np.abs(
                mat_data["u_fe_pod"] - mat_data["u_true_pod"]
            ).reshape(n_sigma, B, Nr, Nt),
            "f_fe_abs_error_grid": np.abs(
                mat_data["f_fe_pod"] - mat_data["f_true_pod"]
            ).reshape(n_sigma, B, Nr, Nt),
            "u_sno_abs_error_grid": np.abs(
                mat_data["u_sno_pod"] - mat_data["u_true_pod"]
            ).reshape(n_sigma, B, Nr, Nt),
        })

    savemat(mat_path, mat_data, do_compression=True)
    print("[Saved]", mat_path)
    return mat_path


mat_path = export_mat(cfg, results, fe_path, ol_path)



## 8. Notebook 内快速可视化

下面默认画第一个 \(\sigma\) 的第一个样本。你可以修改 `i_sigma` 和 `idx`。


In [ ]:

import matplotlib.pyplot as plt

# i_sigma = 0 -> sigma=0.5; i_sigma = 1 -> sigma=1.0; i_sigma = 2 -> sigma=2.0
I_SIGMA = 0
IDX = 0

r = results[I_SIGMA]
coords = to_numpy(r["batch"].pod_coords)

u_true = to_numpy(r["batch"].u_pod[IDX])
u_fe = to_numpy(r["u_fe_pod"][IDX])
u_sno = to_numpy(r["u_sno_pod"][IDX])
err_sno = np.abs(u_sno - u_true)

vmin = min(u_true.min(), u_fe.min(), u_sno.min())
vmax = max(u_true.max(), u_fe.max(), u_sno.max())

plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.scatter(coords[:, 0], coords[:, 1], c=u_true, s=8, vmin=vmin, vmax=vmax)
plt.axis("equal")
plt.colorbar()
plt.title(f"True u, sigma={r['sigma']}")

plt.subplot(1, 4, 2)
plt.scatter(coords[:, 0], coords[:, 1], c=u_fe, s=8, vmin=vmin, vmax=vmax)
plt.axis("equal")
plt.colorbar()
plt.title("FE oracle")

plt.subplot(1, 4, 3)
plt.scatter(coords[:, 0], coords[:, 1], c=u_sno, s=8, vmin=vmin, vmax=vmax)
plt.axis("equal")
plt.colorbar()
plt.title("SNO prediction")

plt.subplot(1, 4, 4)
plt.scatter(coords[:, 0], coords[:, 1], c=err_sno, s=8)
plt.axis("equal")
plt.colorbar()
plt.title("|SNO - true|")

plt.tight_layout()
plt.show()

print("sigma:", r["sigma"])
print("sample idx:", IDX)
print("FE u RL2 on pod:", float(to_numpy(r["err_fe_u_pod"])[IDX]))
print("SNO u RL2 on pod:", float(to_numpy(r["err_sno_u_pod"])[IDX]))
print("latent RL2:", float(to_numpy(r["err_latent"])[IDX]))



## 9. MATLAB 读取示例

```matlab
S = load('low_sigma_fe_sno_diagnostic.mat');
cols = string(S.summary_columns);
T = array2table(S.summary, 'VariableNames', cellstr(cols));
disp(T);

% 画 sigma=1.0 的第 1 个样本
% 若 sigma_values=[0.5;1.0;2.0]，则 i_sigma=2

i_sigma = 2;
idx = 1;

X = squeeze(S.x_grid);
Y = squeeze(S.y_grid);
U_true = squeeze(S.u_true_grid(i_sigma, idx, :, :));
U_fe   = squeeze(S.u_fe_grid(i_sigma, idx, :, :));
U_sno  = squeeze(S.u_sno_grid(i_sigma, idx, :, :));
E_sno  = squeeze(S.u_sno_abs_error_grid(i_sigma, idx, :, :));

figure('Color','w','Position',[100,100,1400,350]);
tiledlayout(1,4,'TileSpacing','compact','Padding','compact');

nexttile; surf(X,Y,U_true,'EdgeColor','none'); view(2); axis equal tight; colorbar; title('True u');
nexttile; surf(X,Y,U_fe,'EdgeColor','none');   view(2); axis equal tight; colorbar; title('FE oracle');
nexttile; surf(X,Y,U_sno,'EdgeColor','none');  view(2); axis equal tight; colorbar; title('SNO prediction');
nexttile; surf(X,Y,E_sno,'EdgeColor','none');  view(2); axis equal tight; colorbar; title('|SNO - true|');
colormap turbo;
```
